# Reel-Out Path Parametrization Demo

This notebook reproduces the defaults from `scripts/optimize_path/reelout_cst.py` and walks through assembling the kite system, defining the path parametrisation, and visualising the resulting trajectory. Run the cells sequentially to generate the baseline quasi-steady solution.

## Prerequisites

- Install the project in editable mode: `pip install -e .[dev]`
- Start Jupyter from the repository root so that relative paths resolve correctly.
- Matplotlib plots are configured to use the house colour palette defined in `awetrim.utils.color_palette`.

In [ ]:
from pathlib import Path
import json
import copy
import numpy as np
import matplotlib.pyplot as plt

from awetrim import SystemModel, State
from awetrim.environment import Wind
from awetrim.system.kite import Kite
from awetrim.system.tether import RigidLumpedTether
from awetrim.timeseries.phase_parametrized import PhaseParameterized
from awetrim.utils.color_palette import set_plot_style_no_latex, get_color_list

set_plot_style_no_latex()


## Load Aerodynamic Input and Wind Model

The LEI-V3 aerodynamic coefficients are stored under `data/LEI-V3-KITE/v3_aero_input.json`. The helper block below resolves the project root even when the notebook is opened from the `parametrize_path` folder.

In [ ]:
project_root = Path.cwd()
if project_root.name == "parametrize_path":
    project_root = project_root.parent
data_path = project_root / "data" / "LEI-V3-KITE" / "v3_aero_input.json"
with data_path.open("r", encoding="utf-8") as handle:
    aero_input = json.load(handle)

wind = Wind(wind_model="uniform", z0=0.1)
wind.speed_wind_ref = 10.0  # m/s at 100 m
wind.speed_friction = 0.41 * wind.speed_wind_ref / np.log(100.0 / wind.z0)
print(f"Friction speed: {wind.speed_friction:.3f} m/s")


## Assemble the Kite System

Instantiate the kite, tether, and `SystemModel`. The mass, area, and steering settings match the reel-out case study.

In [ ]:
kite = Kite(
    mass_wing=14.2,
    area_wing=19.75,
    aero_input=aero_input,
    mass_kcu=10.0,
    steering_control="asymmetric",
)
tether = RigidLumpedTether(diameter=0.01)
system = SystemModel(
    dof=3,
    quasi_steady=True,
    kite=kite,
    tether=tether,
    wind_model=wind,
)
system.input_depower = 0.0

baseline_state = State(
    t=0.0,
    s=np.pi / 2,
    s_dot=2.0,
    length_tether=199.6,
    distance_radial=200.0,
    speed_radial=2.0,
    tension_tether_ground=1e4,
)
baseline_state_dict = baseline_state.to_dict()


## Configure the Reel-Out Pattern

A CST-based Lissajous pattern governs the azimuth and elevation angles while the winch uses a force-controlled reel-out strategy. These parameters match the standalone script and can be tweaked interactively in this notebook.

In [ ]:
pattern_config = {
    "pattern_type": "cst_lissajous",
    "path_parameters": {
        "omega": 1.0,
        "r0": 230.0,
        "az_amp0": 0.4815631965341702,
        "beta_amp0": 0.09875937127714636,
        "width_phi": 0.5,
        "width_beta": 0.5,
        "left_first": True,
        "normalize_bumps": False,
        "repeat_phi": True,
        "repeat_beta": True,
        "beta_coeffs": np.array([0.25485496, -0.99986137, 0.12645635, -0.86821607, 0.35302077]),
        "az_coeffs": [0, 0, 0, 0, 0],
        "kbeta": 0.0,
        "beta0": 0.4414535012239937,
        "kappa": 0.0,
    },
    "radial_parameters": {
        "reeling_strategy": "force",
        "force_model": "quadratic",
        "reeling_speed": 0.0,
        "max_tether_force": 2e4,
        "min_tether_force": 2000.0,
        "softplus": True,
        "softplus_beta": 1e-4,
        "softminus": True,
        "softminus_beta": 1e-3,
        "slope": 2716.0,
        "offset": 0.0,
    },
    "start_time": 0.0,
    "end_time": 35.0,
    "start_angle": float(np.pi / 2),
    "end_angle": float(2 * np.pi + np.pi / 2),
    "n_points": 600,
    "optimization_parameters": [
        "az_amp0",
        "beta_amp0",
        "beta0",
        "beta_coeffs",
    ],
}


## Simulate the Baseline Trajectory

The solver marches along the phase grid (`n_points` samples) using the quasi-steady residual solver at each step.

In [ ]:
baseline_phase = PhaseParameterized(
    system,
    quasi_steady=True,
    pattern_config=pattern_config,
    tension_min=3000.0,
    tension_max=25000.0,
)
baseline_phase.run_simulation_phase(start_state=baseline_state_dict)
print(f"Stored {len(baseline_phase.states)} states.")


## Visualise Key Quantities

Plot the 3D trajectory alongside ground-projected path and selected time histories. The helper returns the Matplotlib figure handles so you can customise colours or save them to disk.

In [ ]:
fig, axes_map, _ = baseline_phase.plot_overview_3d(
    label="QS baseline",
    color=get_color_list()[0],
    linestyle="--",
    variables=["speed_tangential", "tension_tether_ground", "input_steering", "speed_radial"],
    x_param="s",
)
fig.suptitle("Reel-out Lissajous (Baseline)")
plt.tight_layout()
plt.show()


## Optional: Optimise the Path

You can reuse the same configuration to run the built-in optimiser. Uncomment the cell below if you wish to compare the optimised pattern against the baseline. Depending on solver settings this may take several minutes.

In [ ]:
# optimized_phase = PhaseParameterized(
#     system,
#     quasi_steady=True,
#     pattern_config=copy.deepcopy(pattern_config),
#     tension_min=3000.0,
#     tension_max=25000.0,
# )
# optimized_phase.run_simulation_opti_phase(start_state=baseline_state_dict)
# optimized_phase.run_simulation_phase(start_state=baseline_state_dict)
# metrics = optimized_phase.energy_metrics(baseline_phase)
# print(f"Power improvement: {metrics['power_diff_percent']:.2f}%")
# optimized_phase.plot_overview_3d(
#     label="QS opti",
#     color=get_color_list()[1],
#     linestyle="-",
#     variables=["speed_tangential", "tension_tether_ground", "input_steering", "speed_radial"],
#     x_param="s",
#     axes=axes_map,
# )
# plt.show()


## Next Steps

- Adjust the coefficients in `pattern_config['path_parameters']` to explore new trajectories.
- Export `baseline_phase.states` to pandas for reporting or further post-processing.
- Integrate additional constraints or controls by extending `PhaseParameterized` in `src/awetrim/timeseries/phase_parametrized.py`.